# GSE117751 — Autoimmune Retinopathy (Human Immunology Panel)

**Dataset**: [GSE117751](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE117751)
**Panel**: Nanostring nCounter Human Immunology v2 (579 endogenous + 15 housekeeping genes)
**Samples**: 42 whole blood (PAXgene) — 14 AIR (autoimmune retinopathy), 14 RP (retinitis pigmentosa), 14 healthy controls
**Source**: Used in NanoTube package tutorials (comparability reference)

This vignette demonstrates ncountr on a 3-group immunology design — a dataset also used in R package NanoTube tutorials, enabling direct comparison of results.

## 1. Download and Parse

In [ ]:
from ncountr.io.geo import fetch_geo
import ncountr
import pandas as pd

rcc_dir = fetch_geo("GSE117751", output_dir="data")

In [ ]:
exp = ncountr.read_rcc(rcc_dir, sample_id_pattern=r"(GSM\d+)", sample_id_from="filename")

# Assign groups by GSM accession range (from GEO metadata)
# GSM3308226-3308239: AIR, GSM3308240-3308253: RP, GSM3308254-3308267: Control
meta = {}
for s in exp.samples:
    num = int(s[3:])
    if num <= 3308239:
        meta[s] = {"group": "AIR"}
    elif num <= 3308253:
        meta[s] = {"group": "RP"}
    else:
        meta[s] = {"group": "Control"}

exp.sample_meta = pd.DataFrame.from_dict(meta, orient="index").rename_axis("sample")
print(exp)
print(f"\n{exp.sample_meta['group'].value_counts()}")

## 2. QC and Normalization

In [ ]:
qc_results = ncountr.qc(exp)
print(f"FOV pass: {qc_results['FovPass'].sum()}/{len(qc_results)}")
print(f"PosCtrl pass: {qc_results['PosCtrlPass'].sum()}/{len(qc_results)}")

from ncountr.plotting.qc_plots import plot_qc
fig = plot_qc(exp)
fig.tight_layout()

In [ ]:
ncountr.normalize(exp, method="pos_hk")
print(f"Normalized: {exp.normalized.shape[0]} genes x {exp.normalized.shape[1]} samples")

## 3. Differential Expression

### AIR vs Healthy Controls

In [ ]:
air = [s for s in exp.samples if meta[s]["group"] == "AIR"]
rp = [s for s in exp.samples if meta[s]["group"] == "RP"]
ctrl = [s for s in exp.samples if meta[s]["group"] == "Control"]

de_air = ncountr.de(exp, group_a=air, group_b=ctrl, test="mannwhitneyu")
n_sig = (de_air["padj"] < 0.05).sum()
print(f"AIR ({len(air)}) vs Control ({len(ctrl)})")
print(f"Significant (padj < 0.05): {n_sig}")
print(f"\nTop 15 genes by p-value:")
de_air.sort_values("pvalue").head(15)[["gene", "log2FC", "pvalue", "padj"]]

In [ ]:
from ncountr.plotting.de_plots import plot_volcano

fig = plot_volcano(
    de_air,
    highlight_genes=ncountr.get_gene_set("IFN_JAKSTAT"),
    highlight_label="IFN/JAK-STAT",
    title="AIR vs Healthy Controls\n(GSE117751, Human Immunology v2)",
)

### RP vs Healthy Controls

In [ ]:
de_rp = ncountr.de(exp, group_a=rp, group_b=ctrl, test="mannwhitneyu")
n_sig = (de_rp["padj"] < 0.05).sum()
print(f"RP ({len(rp)}) vs Control ({len(ctrl)})")
print(f"Significant (padj < 0.05): {n_sig}")
print(f"\nTop 10 genes by p-value:")
de_rp.sort_values("pvalue").head(10)[["gene", "log2FC", "pvalue", "padj"]]

### AIR vs RP

In [ ]:
de_air_rp = ncountr.de(exp, group_a=air, group_b=rp, test="mannwhitneyu")
n_sig = (de_air_rp["padj"] < 0.05).sum()
print(f"AIR ({len(air)}) vs RP ({len(rp)})")
print(f"Significant (padj < 0.05): {n_sig}")
print(f"\nTop 10 genes by p-value:")
de_air_rp.sort_values("pvalue").head(10)[["gene", "log2FC", "pvalue", "padj"]]

## 4. Heatmap — Top DE Genes Across All Comparisons

In [ ]:
# Union of top genes from all three comparisons
top_air = set(de_air.sort_values("pvalue").head(15)["gene"])
top_rp = set(de_rp.sort_values("pvalue").head(10)["gene"])
top_air_rp = set(de_air_rp.sort_values("pvalue").head(10)["gene"])
top_genes = sorted(top_air | top_rp | top_air_rp)

# Order samples by group
ordered = sorted(exp.samples, key=lambda s: ({"AIR": 0, "RP": 1, "Control": 2}[meta[s]["group"]], s))

from ncountr.plotting.heatmaps import plot_heatmap
fig = plot_heatmap(
    exp,
    genes=top_genes,
    samples=ordered,
    z_score=True,
    title="Top DE genes across all comparisons (z-scored)",
)

## 5. Gene Set Scoring

In [ ]:
scores = ncountr.score_gene_set(exp, gene_set="IFN_JAKSTAT")

score_df = pd.DataFrame({
    "sample": exp.samples,
    "IFN_JAKSTAT_score": [scores[s] for s in exp.samples],
    "group": [meta[s]["group"] for s in exp.samples],
})

from ncountr.plotting.pathway_plots import plot_pathway_scores
fig = plot_pathway_scores(
    score_df,
    score_column="IFN_JAKSTAT_score",
    group_column="group",
    title="IFN/JAK-STAT Pathway Score\n(GSE117751)",
)

## 6. Export to AnnData

In [ ]:
adata = ncountr.to_anndata(exp)
print(adata)

## Summary

| Comparison | Samples | Significant (FDR < 0.05) | Top hits |
|---|---|---|---|
| AIR vs Control | 14 vs 14 | 2 (IKBKB, MIF) | IKBKB, MIF, CR1, IL8, NFKB2, STAT5A |
| RP vs Control | 14 vs 14 | 0 | CCL2, SELE, CASP2, TLR9 |
| AIR vs RP | 14 vs 14 | 0 | IKBKB, C1S, TRAF3, PLA2G2E |

**Key observations:**
- AIR patients show significantly upregulated NF-kB signaling (IKBKB, NFKB2) — consistent with autoimmune activation
- MIF (macrophage migration inhibitory factor) is significantly downregulated in AIR, suggesting altered innate immune regulation
- RP patients show minimal transcriptomic changes vs controls — expected for a non-autoimmune degenerative condition
- AIR vs RP differences highlight complement (C1S) and chemokine (CCL13, CCL15) pathways
- This dataset was also analyzed in the NanoTube R package tutorial, enabling direct cross-tool comparison